# **5. The price advisor**

Chapter 4 gave us a forecast. Chapter 5 turns it into advice a host can act on: when to
raise a price, when to leave it alone, and how to say so in an SMS.

One fact shapes everything in this chapter. **Our data contain no prices.** The synthetic
occupancy was generated without any price response, so we cannot learn or test how guests
react to price. Every claim about revenue rests on a stated assumption, which we vary
instead of defending. The advisor therefore has to be four things: honest about that
assumption, safe when the assumption is wrong, easy to explain, and able to learn from
what hosts accept or reject.

## **5.1 When is a price change safe? A revenue grid**

### **5.1.1 Why**
Before writing any rule, we find out when raising or cutting a price can lose money, and
how that depends on two things we do not know: how sensitive guests are to price, and
how much demand exceeds the rooms.

### **5.1.2 Model**
- **D** is the demand at the current price, as a share of rooms. It can exceed the
  occupancy cap of 98%, meaning more guests would have come than there are rooms.
- A price change of **r** (for example +10%) changes demand to D x (1 + r)^eps, where
  **eps** is the price elasticity, a negative number. eps = -0.6 means guests barely react;
  eps = -2.0 means they react strongly.
- Occupancy is demand, limited to the cap. Revenue is price times occupancy, so the ratio
  of revenue after to revenue before is (1 + r) x occupancy after / occupancy before.
- We do not claim any eps is true. We scan from -0.6 to -2.0.

### **5.1.3 Method**
1. Compute the revenue change on a grid of D (0.60 to 1.40), eps (-0.6, -1.0, -1.4, -2.0)
   and price changes of +5%, +10%, +20% and -10%.
2. Derive the largest safe raise: for D at or above the cap and |eps| above 1, a raise r is
   safe (revenue does not fall) up to r* = (D / cap)^(1 / (|eps| - 1)) - 1. For eps = -2
   this simplifies to D / cap - 1. For D below the cap, no raise is safe when |eps| is
   above 1. Check the formula against a numerical search.

### **5.1.4 Predictions (written before the run)**
1. For D = 0.80 or 0.90, a +10% price rise changes revenue by +3.9%, 0.0%, -3.7% and
   -9.1% for eps = -0.6, -1.0, -1.4 and -2.0.
2. For D = 1.20, a +20% rise changes revenue by +20.0%, +20.0%, +13.8% and +2.0% for the
   same four elasticities.
3. The numerical largest safe raise matches the formula within 0.1 percentage points at
   eps = -2.0 and eps = -1.4.
4. A -10% price cut loses exactly 10.0% for every eps when D is 1.00 or more (the cap
   binds). At D = 0.60 it changes revenue by -4.1%, 0.0%, +4.3% and +11.1%.

### **5.1.5 Decision rules (if the predictions hold)**
- **R1.** Never suggest a price cut by default. A cut helps only if guests are very price
  sensitive and rooms are not close to full, and we cannot know either.
- **R2.** Suggest a raise only where demand is expected to exceed the rooms, and cap it
  at the safe limit for eps = -2, which is D / cap - 1. That raise cannot lose revenue
  for any elasticity up to magnitude 2.
- **R3.** On quiet days suggest non-price actions instead (a promotion, an airtime reward
  for bringing a friend), which the hackathon brief also mentions.
- **R4.** To apply R2 we need an estimate of D, and our forecast is capped at 98%. Step
  5.2 tests whether the model's unclipped prediction can estimate demand above capacity,
  using the generator's known latent demand.

In [1]:
import numpy as np
import pandas as pd

CAP = 0.98


def revenue_ratio(D, r, eps, cap=CAP):
    """Revenue after a price change of r, divided by revenue at the current price.
    D: demand at the current price as a share of rooms (it can exceed the cap).
    eps: price elasticity of demand (negative). Occupancy can never exceed cap."""
    occ_now = np.minimum(D, cap)
    occ_new = np.minimum(D * (1 + r) ** eps, cap)
    return (1 + r) * occ_new / occ_now


D_grid = np.array([0.60, 0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20, 1.40])
eps_grid = np.array([-0.6, -1.0, -1.4, -2.0])

for r in [0.05, 0.10, 0.20, -0.10]:
    grid = pd.DataFrame(
        100 * (revenue_ratio(D_grid[:, None], r, eps_grid[None, :]) - 1),
        index=[f"D={d:.2f}" for d in D_grid],
        columns=[f"eps={e}" for e in eps_grid],
    )
    print(f"Revenue change (%) for a price change of {r:+.0%}:")
    print(grid.round(1).to_string())
    print()


def breakeven_numeric(D, eps, cap=CAP, step=0.0005, rmax=1.5):
    """Largest raise r for which revenue does not fall, found by search."""
    rs = np.arange(0.0, rmax + step, step)
    ok = revenue_ratio(D, rs, eps, cap) >= 1 - 1e-12
    if ok.all():
        return rmax
    first_bad = int(np.argmax(~ok))
    return float(rs[first_bad - 1]) if first_bad > 0 else 0.0


def breakeven_formula(D, eps, cap=CAP):
    if D < cap:
        return 0.0
    return (D / cap) ** (1 / (abs(eps) - 1)) - 1


rows = []
for D in [0.80, 0.90, 0.98, 1.00, 1.05, 1.10, 1.20, 1.30]:
    rows.append({
        "D": D,
        "eps -2.0 numeric": breakeven_numeric(D, -2.0),
        "eps -2.0 formula": breakeven_formula(D, -2.0),
        "eps -1.4 numeric": breakeven_numeric(D, -1.4),
        "eps -1.4 formula": breakeven_formula(D, -1.4),
    })
be = pd.DataFrame(rows).set_index("D")
print("Largest safe price rise (%), numeric search against the formula:")
print((100 * be).round(1).to_string())
for e in ("-2.0", "-1.4"):
    gap = (100 * (be[f"eps {e} numeric"] - be[f"eps {e} formula"])).abs().max()
    print(f"largest gap at eps {e}: {gap:.2f} percentage points")

Revenue change (%) for a price change of +5%:
        eps=-0.6  eps=-1.0  eps=-1.4  eps=-2.0
D=0.60       2.0       0.0      -1.9      -4.8
D=0.80       2.0       0.0      -1.9      -4.8
D=0.90       2.0       0.0      -1.9      -4.8
D=0.95       2.0       0.0      -1.9      -4.8
D=1.00       4.1       2.0       0.1      -2.8
D=1.05       5.0       5.0       5.0       2.0
D=1.10       5.0       5.0       5.0       5.0
D=1.20       5.0       5.0       5.0       5.0
D=1.40       5.0       5.0       5.0       5.0

Revenue change (%) for a price change of +10%:
        eps=-0.6  eps=-1.0  eps=-1.4  eps=-2.0
D=0.60       3.9       0.0      -3.7      -9.1
D=0.80       3.9       0.0      -3.7      -9.1
D=0.90       3.9       0.0      -3.7      -9.1
D=0.95       3.9       0.0      -3.7      -9.1
D=1.00       6.0       2.0      -1.8      -7.2
D=1.05      10.0       7.1       3.1      -2.6
D=1.10      10.0      10.0       8.0       2.0
D=1.20      10.0      10.0      10.0      10.0
D=1.40      1

## **5.2 Review of results: the revenue grid (step 5.1)**

### **5.2.1 Verdict**
All four predictions held exactly, so the grid supports decision rules R1 to R3. It also
showed something I had not planned: the break-even raise (D / cap - 1) gains nothing in the
worst case, so the advisor should suggest a smaller raise inside it. Rule R4 (can we
estimate demand above the rooms?) is still open and is the subject of step 5.3.

### **5.2.2 Prediction scorecard**

| Prediction (5.1.4) | Result | Verdict |
|---|---|---|
| D = 0.80 or 0.90, +10%: +3.9, 0.0, -3.7, -9.1 | Exactly those values in both rows | Held |
| D = 1.20, +20%: +20.0, +20.0, +13.8, +2.0 | Exactly those values | Held |
| Numeric safe raise matches the formula within 0.1 points | Largest gaps 0.05 (eps -2.0) and 0.03 (eps -1.4) | Held |
| A -10% cut loses exactly 10.0% for D of 1.00 or more; at D = 0.60 it gives -4.1, 0.0, +4.3, +11.1 | Exactly those values | Held |

### **5.2.3 What the numbers show**

#### **5.2.3.1 Below capacity, a raise is a bet on insensitive guests**
For every D from 0.60 to 0.95 the result is the same, because occupancy is not limited by
the rooms. A +5% raise gives +2.0%, 0.0%, -1.9% and -4.8% at eps = -0.6, -1.0, -1.4 and
-2.0; +10% gives +3.9%, 0.0%, -3.7%, -9.1%; +20% gives +7.6%, 0.0%, -7.0%, -16.7%. The sign
flips at eps = -1: a raise helps only if guests react less than proportionally.

#### **5.2.3.2 Above capacity, a raise is safe up to the excess**

| D (demand as a share of rooms) | Largest safe raise at eps = -2.0 | at eps = -1.4 |
|---|---|---|
| 1.00 | 2.0% | 5.2% |
| 1.05 | 7.1% | 18.8% |
| 1.10 | 12.2% | 33.4% |
| 1.20 | 22.4% | 65.9% |
| 1.30 | 32.6% | 102.6% |

At eps = -2 the limit is exactly D / cap - 1, and the formula agrees with the numerical
search to within 0.05 points. The limit is much larger for milder elasticities.

#### **5.2.3.3 Cuts help only in a narrow zone**
At D = 0.60 a -10% cut gains +4.3% (eps = -1.4) and +11.1% (eps = -2.0); at D = 0.80 it
gains +4.3% and +10.3%. By D = 0.90 the cap binds after the cut and the cut loses 2.0% for
every eps of -1.0 or more negative (-4.1% at eps = -0.6). At D = 0.95 it loses 7.2% for every
eps, and at D of 1.00 or more it loses exactly 10%. A cut pays only for very sensitive
guests and well below capacity, and we cannot tell either from our data.

#### **5.2.3.4 The gamble zone**
For D between about 0.90 and 1.00, neither move is safe: raises lose if guests are
sensitive (eps below -1), and cuts lose because the rooms fill anyway. The advisor holds
there.

#### **5.2.3.5 A refinement: how big should a raise be?**
The formula D / cap - 1 is the break-even raise, which gains nothing at eps = -2, so it is
too aggressive to suggest. Three natural sizes (computed from the formulas):

| D | Keeps rooms full at eps = -2: (D/cap)^(1/2) - 1 | Best at eps = -1.4: (D/cap)^(1/1.4) - 1 | Break-even at eps = -2: D/cap - 1 |
|---|---|---|---|
| 1.10 | 5.9% | 8.6% | 12.2% |
| 1.20 | 10.7% | 15.6% | 22.4% |

I choose the middle one, **(D/cap)^(1/1.4) - 1, rounded down to a multiple of 5%**, with a
minimum of 5% and a maximum of 25%. It is never above the break-even at eps = -2, so
it loses nothing for any elasticity up to magnitude 2. The elasticity of -1.4 is the middle
of the scanned range, an assumption and not an estimate. A 5% suggestion needs D of about
1.05 or more.

### **5.2.4 Limits**
- Constant elasticity is a simplification, and the range -2.0 to -0.6 is assumed.
- The model counts revenue only. It ignores costs per occupied room, competitors and
  guests' memory of past prices.
- In practice D is unknown. We have only a forecast capped at the rooms available.

### **5.2.5 Decision**
Adopt R1 (no default cuts), R2 (raise only where demand exceeds rooms, sized by the
refined formula) and R3 (non-price actions on quiet days). Test R4 next.

## **5.3 Next step: can we see demand above the rooms?**

### **5.3.1 Why**
R2 needs D, demand as a share of rooms, which can exceed 100%. Our occupancy forecast stops
at the rooms available, and the model was trained on occupancy that is itself capped, so
it never saw demand above the cap. We test whether the model's **unclipped** prediction is
a usable estimate of D, using the generator's own latent demand as the truth.

### **5.3.2 Method**
1. Add unclipped columns to the forecaster: `demand`, `demand_low`, `demand_high` (Part A).
2. Refit on all history and forecast 146 days (to 31 March 2027) for each destination.
3. Compare the unclipped prediction (D_hat) with the generator's noise-free latent demand
   (D_true), on days where D_true exceeds the 98% cap.
4. Apply the advisor's raise rule to D_hat (rule A) and to the lower bound of the demand
   interval (rule B), and judge each against what D_true says is safe.

### **5.3.3 The raise rule used**
For D above the cap, raise by (D / cap)^(1 / 1.4) - 1, rounded down to a multiple of 5%,
with a minimum of 5% and a maximum of 25%. Otherwise no raise.

### **5.3.4 Predictions (written before the run)**
1. Days in the 146 where the true demand is above the cap: Naivasha 3 to 6, Diani 5 to 14,
   Lamu 3 to 12, Maasai Mara 0 to 4, Amboseli 0 to 2.
2. On those days D_hat is below D_true: the ratio D_hat / D_true is between 0.85 and 0.97 in
   every destination that has such days.
3. Rule A is safe: at least 95% of its suggestions are at or below the break-even computed
   from the true demand at eps = -2, and the mean revenue change on suggested days is not
   negative at eps = -2.0 and eps = -1.4.
4. Rule A is cautious to the point of silence: overall it finds fewer than half of the
   opportunity days, meaning the days where the same rule applied to the true demand would
   suggest a raise.
5. Rule B suggests a raise on no more days than rule A, and is also at least 95% safe.

### **5.3.5 Decision rule**
- If rule A is safe but finds fewer than half of the opportunities, test a censoring-aware
  fit in 5.4 (observations at the cap treated as "at least this much", not as exact).
- If rule A finds half or more and is safe, adopt it, and keep rule B as a conservative
  option.
- If rule A is not safe (under 95%), adopt rule B and find out why D_hat is too high.

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
import locat.forecasting.features as ffeat
import locat.forecasting.model as fmodel
import locat.forecasting.synthetic as synth

CAP = 0.98
out_dir = Path("../data/processed")
DESTINATIONS = ["Maasai Mara", "Naivasha", "Diani", "Lamu", "Amboseli"]
HISTORY_END = pd.Timestamp("2026-11-05")
FUTURE_END = pd.Timestamp("2027-03-31")


def revenue_ratio(D, r, eps, cap=CAP):
    occ_now = np.minimum(D, cap)
    occ_new = np.minimum(D * (1 + r) ** eps, cap)
    return (1 + r) * occ_new / occ_now


def suggested_raise(D, cap=CAP, eps_mid=-1.4, step=0.05, min_raise=0.05, max_raise=0.25):
    """Best raise at the middle elasticity; never above the break-even at eps = -2."""
    D = np.asarray(D, dtype=float)
    r = np.where(D > cap, (D / cap) ** (1 / abs(eps_mid)) - 1, 0.0)
    r = np.floor(r / step + 1e-9) * step
    return np.where(r < min_raise, 0.0, np.minimum(r, max_raise))


# ---- setup: data, forecaster, truth ----
cal = pd.read_csv(out_dir / "calendar_daily.csv", parse_dates=["date"])
seasonal = pd.read_csv(out_dir / "seasonal_index.csv")
idx_map = dict(zip(seasonal["month_number"], seasonal["seasonal_index"]))
feats = ffeat.build_features(cal, idx_map)
q = pd.read_csv(out_dir / "forecast_interval_quantiles.csv", index_col=0)
syn = pd.read_csv(out_dir / "synthetic_occupancy.csv", parse_dates=["date"])
occ = syn.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]

assump = json.loads((out_dir / "synthetic_assumptions.json").read_text())
params = pd.DataFrame(assump["params"])
month_adjust = pd.DataFrame.from_dict(assump["month_adjust"], orient="index")
settings = assump["globals"]

fc = fmodel.OccupancyForecaster(feats, q).fit(occ)
truth = synth.generate_history(
    cal, params, month_adjust, dict(settings, noise_sd=0.0), idx_map, FUTURE_END,
    normalise_until=HISTORY_END,
)
truth_latent = truth.pivot(index="date", columns="destination", values="latent_demand")[DESTINATIONS]

# ---- forecasts of demand, side by side with the truth ----
horizon = (FUTURE_END - fc.last_date).days
parts = []
for dest in DESTINATIONS:
    f = fc.forecast(dest, horizon)
    assert np.allclose(f["forecast"], np.clip(f["demand"], 0.01, CAP))  # old columns unchanged
    parts.append(pd.DataFrame({
        "destination": dest, "date": f.index,
        "D_hat": f["demand"].to_numpy(), "D_low": f["demand_low"].to_numpy(),
        "D_true": truth_latent.loc[f.index, dest].to_numpy(),
    }))
d = pd.concat(parts, ignore_index=True)
d["excess"] = d["D_true"] > CAP
print(f"compared {horizon} days per destination: {d['date'].min().date()} to {d['date'].max().date()}\n")

# 1) Days with true demand above the rooms, and how well D_hat sees them
ex = d[d["excess"]].groupby("destination").agg(
    days=("D_true", "size"),
    mean_D_true=("D_true", "mean"),
    mean_D_hat=("D_hat", "mean"),
    share_D_hat_above_cap=("D_hat", lambda s: (s > CAP).mean()),
).reindex(DESTINATIONS)
ex["days"] = ex["days"].fillna(0).astype(int)
ex["D_hat_over_D_true"] = ex["mean_D_hat"] / ex["mean_D_true"]
print("Days where true demand exceeds the rooms:")
print(ex.round(2).to_string())

# 2) The raise rule applied to D_hat (rule A) and to its lower bound (rule B)
EPS = [-2.0, -1.4, -1.0]
d["r_true_rule"] = suggested_raise(d["D_true"])            # the rule with perfect knowledge
d["breakeven_true"] = np.maximum(d["D_true"] / CAP - 1, 0.0)
rows = []
for rule, col in [("A_point", "D_hat"), ("B_lower_bound", "D_low")]:
    d[f"r_{rule}"] = suggested_raise(d[col])
    for dest in DESTINATIONS + ["ALL"]:
        s = d if dest == "ALL" else d[d["destination"] == dest]
        sugg = s[f"r_{rule}"] > 0
        opp = s["r_true_rule"] > 0
        row = {
            "rule": rule, "destination": dest,
            "suggested_days": int(sugg.sum()),
            "opportunity_days": int(opp.sum()),
            "hit_days": int((sugg & opp).sum()),
            "safe_share_%": (100 * float((s.loc[sugg, f"r_{rule}"] <= s.loc[sugg, "breakeven_true"] + 1e-9).mean())
                             if sugg.any() else np.nan),
        }
        for e in EPS:
            chg = 100 * (revenue_ratio(s.loc[sugg, "D_true"].to_numpy(),
                                       s.loc[sugg, f"r_{rule}"].to_numpy(), e) - 1)
            row[f"rev_change_eps{e}_%"] = chg.mean() if sugg.any() else np.nan
        rows.append(row)
print("\nRaise rule judged against the true demand:")
print(pd.DataFrame(rows).round(1).to_string(index=False))

# 3) A concrete look at the festive days
def show(dest, start, end):
    s = d[(d["destination"] == dest) & (d["date"] >= start) & (d["date"] <= end)]
    print(f"\n{dest}, {start} to {end}:")
    print(pd.DataFrame({
        "day": s["date"].dt.strftime("%a %d %b").to_numpy(),
        "D_true": s["D_true"].round(2).to_numpy(), "D_hat": s["D_hat"].round(2).to_numpy(),
        "D_low": s["D_low"].round(2).to_numpy(),
        "raise_if_known_%": (100 * s["r_true_rule"]).round(0).astype(int).to_numpy(),
        "raise_A_%": (100 * s["r_A_point"]).round(0).astype(int).to_numpy(),
        "raise_B_%": (100 * s["r_B_lower_bound"]).round(0).astype(int).to_numpy(),
    }).to_string(index=False))

show("Naivasha", "2026-12-24", "2026-12-28")
show("Naivasha", "2026-12-31", "2027-01-03")
show("Diani", "2026-12-22", "2026-12-31")

compared 146 days per destination: 2026-11-06 to 2027-03-31

Days where true demand exceeds the rooms:
             days  mean_D_true  mean_D_hat  share_D_hat_above_cap  D_hat_over_D_true
destination                                                                         
Maasai Mara     0          NaN         NaN                    NaN                NaN
Naivasha        4         1.06        0.95                    0.0               0.90
Diani           4         1.10        1.03                    1.0               0.94
Lamu            0          NaN         NaN                    NaN                NaN
Amboseli        0          NaN         NaN                    NaN                NaN

Raise rule judged against the true demand:
         rule destination  suggested_days  opportunity_days  hit_days  safe_share_%  rev_change_eps-2.0_%  rev_change_eps-1.4_%  rev_change_eps-1.0_%
      A_point Maasai Mara               0                 0         0           NaN                   NaN   